In [ ]:
import os
print(os.listdir('/content'))

['.config', '25novog.pdf', '25julog.pdf', '1febog.pdf', '8augog.pdf', 'sample_data', '31janog.pdf', '2augog.pdf', '9augog.pdf', '26julog.pdf', '27novog.pdf', '23julog.pdf']


In [ ]:
import os

for dirname, _, filenames in os.walk('/content'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/content/25novog.pdf
/content/25julog.pdf
/content/1febog.pdf
/content/8augog.pdf
/content/31janog.pdf
/content/2augog.pdf
/content/9augog.pdf
/content/26julog.pdf
/content/27novog.pdf
/content/23julog.pdf
/content/.config/config_sentinel
/content/.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
/content/.config/gce
/content/.config/default_configs.db
/content/.config/.last_survey_prompt.yaml
/content/.config/.last_update_check.json
/content/.config/.last_opt_in_prompt.yaml
/content/.config/active_config
/content/.config/logs/2026.03.30/13.34.38.846859.log
/content/.config/logs/2026.03.30/13.34.25.289195.log
/content/.config/logs/2026.03.30/13.34.26.856841.log
/content/.config/logs/2026.03.30/13.34.14.816231.log
/content/.config/logs/2026.03.30/13.34.38.057647.log
/content/.config/logs/2026.03.30/13.33.51.299064.log
/content/.config/configurations/config_default
/content/sample_data/anscombe.json
/content/sample_data/README.md


In [ ]:
!pip install pdfplumber tqdm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 106.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 130.3 MB/s eta 0:00:00


In [ ]:
import pdfplumber
import json
import os
import re
from pathlib import Path
from tqdm import tqdm

# ── Config ─────────────────────────────────────────
OUTPUT_FILE   = Path("/content/chunks.json")
CHUNK_SIZE    = 800
CHUNK_OVERLAP = 150

# ── Your actual Colab PDF paths ────────────────────
PDF_FILES = [
    {"path": "/content/31janog.pdf", "date": "31 January 2009", "month": "January", "year": "2009", "session": "Budget Session"},
    {"path": "/content/1febog.pdf", "date": "1 February 2009", "month": "February", "year": "2009", "session": "Budget Session"},
    {"path": "/content/23julog.pdf", "date": "23 July 2009", "month": "July", "year": "2009", "session": "Monsoon Session"},
    {"path": "/content/25julog.pdf", "date": "25 July 2009", "month": "July", "year": "2009", "session": "Monsoon Session"},
    {"path": "/content/26julog.pdf", "date": "26 July 2009", "month": "July", "year": "2009", "session": "Monsoon Session"},
    {"path": "/content/2augog.pdf", "date": "2 August 2009", "month": "August", "year": "2009", "session": "Monsoon Session"},
    {"path": "/content/8augog.pdf", "date": "8 August 2009", "month": "August", "year": "2009", "session": "Monsoon Session"},
    {"path": "/content/9augog.pdf", "date": "9 August 2009", "month": "August", "year": "2009", "session": "Monsoon Session"},
    {"path": "/content/25novog.pdf", "date": "25 November 2009", "month": "November", "year": "2009", "session": "Winter Session"},
    {"path": "/content/27novog.pdf", "date": "27 November 2009", "month": "November", "year": "2009", "session": "Winter Session"},
]

# ── Helpers ────────────────────────────────────────

def clean_text(text):
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]{2,}', ' ', text)
    text = re.sub(r'[_]{5,}', '', text)
    text = re.sub(r'\x0c', '', text)
    return text.strip()

def sliding_window_chunks(text, chunk_size, overlap):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

# ── Main Function ──────────────────────────────────

def extract_all_pdfs():
    all_chunks = []
    chunk_id = 0

    for pdf_info in tqdm(PDF_FILES):
        pdf_path = pdf_info["path"]
        filename = Path(pdf_path).name

        if not Path(pdf_path).exists():
            print(f"❌ Not found: {pdf_path}")
            continue

        print(f"\n📄 Processing: {filename}")

        with pdfplumber.open(pdf_path) as pdf:
            total_pages = len(pdf.pages)

            for page_num, page in enumerate(pdf.pages, start=1):
                raw = page.extract_text()

                if not raw or len(raw.strip()) < 50:
                    continue

                cleaned = clean_text(raw)
                page_chunks = sliding_window_chunks(cleaned, CHUNK_SIZE, CHUNK_OVERLAP)

                for i, chunk_text in enumerate(page_chunks):
                    if len(chunk_text.strip()) < 80:
                        continue

                    all_chunks.append({
                        "id": chunk_id,
                        "text": chunk_text,
                        "metadata": {
                            "filename": filename,
                            "date": pdf_info["date"],
                            "month": pdf_info["month"],
                            "year": pdf_info["year"],
                            "session": pdf_info["session"],
                            "page": page_num,
                            "total_pages": total_pages,
                            "chunk_index": i
                        }
                    })
                    chunk_id += 1

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(all_chunks, f, indent=2, ensure_ascii=False)

    print(f"\n✅ Done! {chunk_id} chunks saved to {OUTPUT_FILE}")
    return all_chunks

# ── Run ────────────────────────────────────────────
chunks = extract_all_pdfs()

  0%|          | 0/10 [00:00<?, ?it/s]


📄 Processing: 31janog.pdf


 10%|█         | 1/10 [00:04<00:39,  4.43s/it]


📄 Processing: 1febog.pdf


 20%|██        | 2/10 [00:07<00:28,  3.62s/it]

...


📄 Processing: 25novog.pdf


 90%|█████████ | 9/10 [02:18<00:16, 16.11s/it]


📄 Processing: 27novog.pdf


100%|██████████| 10/10 [02:23<00:00, 14.36s/it]


✅ Done! 4686 chunks saved to /content/chunks.json


In [ ]:
# Cell 3 — verify it worked
import json

with open("/content/chunks.json") as f:
    chunks = json.load(f)

print(f"Total chunks: {len(chunks)}")
print(chunks[0])  # preview first chunk

Total chunks: 4686
{'id': 0, 'text': 'Eighteenth Series, Vol. VI No.1 Friday, January 31, 2025\nMagha 11, 1946 (Saka)\nLOK SABHA DEBATES\n(English Version)\nFourth Session\n(Eighteenth Lok Sabha)\n(Vol. VI contains Nos. 1 to 10)\nLOK SABHA SECRETARIAT\nNEW DELHI', 'metadata': {'filename': '31janog.pdf', 'date': '31 January 2009', 'month': 'January', 'year': '2009', 'session': 'Budget Session', 'page': 1, 'total_pages': 69, 'chunk_index': 0}}


In [ ]:
import json
import re
import pdfplumber
from pathlib import Path

# ✅ Change 1: chunks file path
CHUNKS_FILE = Path("/content/chunks.json")

# ✅ Change 2: your actual PDF paths
PDF_FILES = [
    "/content/31janog.pdf",
    "/content/1febog.pdf",
    "/content/23julog.pdf",
    "/content/25julog.pdf",
    "/content/26julog.pdf",
    "/content/2augog.pdf",
    "/content/8augog.pdf",
    "/content/9augog.pdf",
    "/content/25novog.pdf",
    "/content/27novog.pdf",
]

# ── Metadata Extraction ────────────────────────────

def extract_real_metadata(pdf_path: str) -> dict:
    filename = Path(pdf_path).name
    result = {
        "filename": filename,
        "date": "unknown",
        "day": "unknown",
        "month": "unknown",
        "year": "unknown",
        "session": "unknown",
        "lok_sabha": "unknown",
        "source": "Lok Sabha Proceedings"
    }

    try:
        with pdfplumber.open(pdf_path) as pdf:
            header_text = ""
            for page in pdf.pages[:2]:
                t = page.extract_text()
                if t:
                    header_text += t + "\n"

        year_match = re.search(r'\b(19|20)\d{2}\b', header_text)
        if year_match:
            result["year"] = year_match.group()

        date_match = re.search(
            r'(?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday),?\s*'
            r'([A-Za-z]+)\s+(\d{1,2}),?\s*((?:19|20)\d{2})',
            header_text, re.IGNORECASE
        )

        if date_match:
            month_str = date_match.group(1).capitalize()
            day_str = date_match.group(2)
            year_str = date_match.group(3)

            result["date"] = f"{day_str} {month_str} {year_str}"
            result["day"] = day_str
            result["month"] = month_str
            result["year"] = year_str
        else:
            date_match2 = re.search(
                r'(\d{1,2})\s+(January|February|March|April|May|June|July|'
                r'August|September|October|November|December)\s*,?\s*((?:19|20)\d{2})',
                header_text, re.IGNORECASE
            )
            if date_match2:
                result["day"] = date_match2.group(1)
                result["month"] = date_match2.group(2).capitalize()
                result["year"] = date_match2.group(3)
                result["date"] = f"{result['day']} {result['month']} {result['year']}"

        # Session detection
        month_lower = result["month"].lower()
        if month_lower in ("january","february","march","april","may"):
            result["session"] = "Budget Session"
        elif month_lower in ("june","july","august"):
            result["session"] = "Monsoon Session"
        elif month_lower in ("november","december"):
            result["session"] = "Winter Session"
        else:
            result["session"] = "Special Session"

        # Lok Sabha detection
        ls_match = re.search(
            r'(First|Second|Third|Fourth|Fifth|Sixth|Seventh|Eighth|Ninth|Tenth|'
            r'Eleventh|Twelfth|Thirteenth|Fourteenth|Fifteenth|Sixteenth|'
            r'Seventeenth|Eighteenth|Nineteenth|Twentieth)\s+Lok\s+Sabha',
            header_text, re.IGNORECASE
        )
        if ls_match:
            result["lok_sabha"] = ls_match.group(0)

    except Exception as e:
        print(f"❌ Error in {filename}: {e}")

    return result


# ── Run Metadata Fix ───────────────────────────────

print("🔍 Detecting metadata...\n")
real_metadata = {}

for pdf_path in PDF_FILES:
    filename = Path(pdf_path).name
    meta = extract_real_metadata(pdf_path)
    real_metadata[filename] = meta

    print(f"{filename}")
    print(f"  Date    : {meta['date']}")
    print(f"  Session : {meta['session']}")
    print(f"  Sabha   : {meta['lok_sabha']}\n")


# ── Update chunks.json ─────────────────────────────

print("📝 Updating chunks.json...\n")

with open(CHUNKS_FILE, "r", encoding="utf-8") as f:
    chunks = json.load(f)

for chunk in chunks:
    fname = chunk["metadata"]["filename"]
    if fname in real_metadata:
        correct = real_metadata[fname]
        chunk["metadata"].update({
            "date": correct["date"],
            "day": correct["day"],
            "month": correct["month"],
            "year": correct["year"],
            "session": correct["session"],
            "lok_sabha": correct["lok_sabha"],
        })

with open(CHUNKS_FILE, "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)

print(f"✅ Metadata fixed for {len(chunks)} chunks → {CHUNKS_FILE}")

🔍 Detecting metadata...

31janog.pdf
  Date    : 31 January 2025
  Session : Budget Session
  Sabha   : Eighteenth Lok Sabha

1febog.pdf
  Date    : 01 February 2025
  Session : Budget Session
  Sabha   : Eighteenth Lok Sabha

23julog.pdf
  Date    : 23 July 2024
  Session : Monsoon Session
  Sabha   : Eighteenth Lok Sabha

25julog.pdf
  Date    : 25 July 2024
  Session : Monsoon Session
  Sabha   : Eighteenth Lok Sabha

26julog.pdf
  Date    : 26 July 2024
  Session : Monsoon Session
  Sabha   : Eighteenth Lok Sabha

2augog.pdf
  Date    : 2 August 2024
  Session : Monsoon Session
  Sabha   : Eighteenth Lok Sabha

8augog.pdf
  Date    : 8 August 2024
  Session : Monsoon Session
  Sabha   : Eighteenth Lok Sabha

9augog.pdf
  Date    : 9 August 2024
  Session : Monsoon Session
  Sabha   : Eighteenth Lok Sabha

25novog.pdf
  Date    : 25 November 2024
  Session : Winter Session
  Sabha   : Eighteenth Lok Sabha

27novog.pdf
  Date    : 27 November 2024
  Session : Winter Session
  Sabha  

In [ ]:
!pip install faiss-cpu sentence-transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 74.5 MB/s eta 0:00:00


In [ ]:
import json
import numpy as np
import faiss
import pickle
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

# ── Config (UPDATED FOR COLAB) ─────────────────────
CHUNKS_FILE   = Path("/content/chunks.json")
INDEX_FILE    = Path("/content/faiss.index")
META_FILE     = Path("/content/metadata.pkl")

EMBED_MODEL   = "all-MiniLM-L6-v2"
BATCH_SIZE    = 64


# ── Load chunks ────────────────────────────────────

def load_chunks():
    if not CHUNKS_FILE.exists():
        raise FileNotFoundError(
            f"{CHUNKS_FILE} not found. Run Step 1 first."
        )
    with open(CHUNKS_FILE, "r", encoding="utf-8") as f:
        return json.load(f)


# ── Embed ──────────────────────────────────────────

def embed_chunks(chunks, model):
    texts = [c["text"] for c in chunks]
    print(f"Embedding {len(texts)} chunks...")

    all_embeddings = []
    for i in tqdm(range(0, len(texts), BATCH_SIZE)):
        batch = texts[i : i + BATCH_SIZE]
        vecs = model.encode(batch, normalize_embeddings=True)
        all_embeddings.append(vecs)

    return np.vstack(all_embeddings).astype("float32")


# ── Build FAISS index ──────────────────────────────

def build_faiss_index(embeddings):
    dim = embeddings.shape[1]
    print(f"Building FAISS index with dim={dim}...")

    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    print(f"→ {index.ntotal} vectors indexed")
    return index


# ── Run Pipeline ───────────────────────────────────

chunks = load_chunks()
print(f"Loaded {len(chunks)} chunks")

print("Loading embedding model...")
model = SentenceTransformer(EMBED_MODEL)

embeddings = embed_chunks(chunks, model)

index = build_faiss_index(embeddings)

# Save index
faiss.write_index(index, str(INDEX_FILE))
print(f"✅ Saved index → {INDEX_FILE}")

# Save metadata
metadata = [c["metadata"] | {"text": c["text"]} for c in chunks]
with open(META_FILE, "wb") as f:
    pickle.dump(metadata, f)

print(f"✅ Saved metadata → {META_FILE}")

Loaded 4686 chunks
Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding 4686 chunks...


100%|██████████| 74/74 [00:09<00:00,  7.52it/s]

Building FAISS index with dim=384...
→ 4686 vectors indexed
✅ Saved index → /content/faiss.index
✅ Saved metadata → /content/metadata.pkl


In [ ]:
!pip install streamlit pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 137.5 MB/s eta 0:00:00


In [ ]:
import subprocess
import threading
import time
from pyngrok import ngrok

# 🔑 Set your ngrok auth token
ngrok.set_auth_token("3BlLIAA9dPkvOWH0bq37A3Chmtz_2AZxXFmaAQS9EdrYQygWu")

# ── Save retriever.py ──────────────────────────────
retriever_code = """
# paste your retriever.py code here
"""
with open("/content/retriever.py", "w") as f:
    f.write(retriever_code)

# ── Save app.py ────────────────────────────────────
app_code = """
# paste your app.py code here
"""
with open("/content/app.py", "w") as f:
    f.write(app_code)


# ── Run Streamlit ──────────────────────────────────
def run_streamlit():
    subprocess.run([
        "streamlit", "run", "/content/app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--browser.serverAddress", "0.0.0.0"
    ])

thread = threading.Thread(target=run_streamlit)
thread.daemon = True
thread.start()

time.sleep(8)  # give it time to start


# ── Start ngrok tunnel ─────────────────────────────
public_url = ngrok.connect(8501)
print(f"\n✅ App is live at: {public_url}")


✅ App is live at: NgrokTunnel: "https://maranda-cinerary-muddlingly.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
!streamlit run /content/app.py




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8502
  Network URL: http://172.28.0.12:8502
  External URL: http://34.83.185.242:8502

  Stopping...
